# Task 1 – N-Queens: Propositional Logic (SAT) & Min-Conflicts

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from nqueens.models import Board
from nqueens.constraints import ConstraintDiscoverer, CNFConverter
from nqueens.sat_solver import SATQueensSolver
from nqueens.min_conflicts import MinConflictsSolver, MaxConflictMinConflictsSolver
from nqueens.benchmark import Benchmark

def draw(queens, title=""):
    n = len(queens)
    fig, ax = plt.subplots(figsize=(5, 5))
    for r in range(n):
        for c in range(n):
            ax.add_patch(Rectangle((c, n-1-r), 1, 1, color="#f0d9b5" if (r+c)%2==0 else "#b58863"))
    for c, r in enumerate(queens):
        ax.text(c+.5, n-1-r+.5, "♛", ha="center", va="center", fontsize=max(6, 200//n), color="#1b5e20")
    ax.set_xlim(0, n); ax.set_ylim(0, n); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title)
    plt.show()

## 1. Mô hình hóa: ràng buộc → CNF (ví dụ N = 4)

In [ ]:
b = Board(4)
cons = ConstraintDiscoverer(b).discover()
clauses = CNFConverter(b).convert(cons)
print("Số ràng buộc:", len(cons), "| Số mệnh đề CNF:", len(clauses))
print("Ví dụ 5 mệnh đề đầu:", clauses[:5])

## 2. SAT với Glucose3

In [ ]:
res = SATQueensSolver().solve(8)
print(res.solved, res.num_vars, res.num_clauses, f"{res.time_s:.4f}s")
if res.queens is not None:
    print(Board.to_text(res.queens))
    draw(res.queens, "SAT (Glucose3) - N=8")
else:
    print("Solver không tìm ra nghiệm!")

In [ ]:
sizes = [4, 8, 12, 16, 20, 24, 32, 40, 48]
Benchmark().run_sat(sizes)

In [ ]:
rs = [SATQueensSolver().solve(n) for n in sizes]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(sizes, [r.num_clauses for r in rs], "o-"); ax[0].set_title("#clauses vs N"); ax[0].set_xlabel("N")
ax[1].plot(sizes, [r.time_s for r in rs], "o-r"); ax[1].set_title("Solve time (s) vs N"); ax[1].set_xlabel("N")
plt.tight_layout(); plt.show()

## 3. Min-Conflicts: chuẩn vs heuristic MaxConflict

In [ ]:
for S in (MinConflictsSolver, MaxConflictMinConflictsSolver):
    r = S(seed=1).solve(12)
    print(r.solver, "| solved:", r.solved, "| steps:", r.steps, f"| {r.time_s:.4f}s")
    if r.queens is not None:
        draw(r.queens, f"{r.solver} - N=12")
    else:
        print(f"{r.solver} không tìm ra nghiệm với N=12 (thử seed khác)")

In [ ]:
Benchmark().run_min_conflicts([8, 16, 32, 64, 100, 200, 500], runs=10, seed=0)